In [82]:
import pandas as pd
import numpy as np
import ast

In [42]:
anime_df = pd.read_csv("archive-2/animes.csv")

In [43]:
anime_df.head()

,title,anime_id,url,image_path,airing_status,num_episodes,mpaa_rating,last_scraped_date,title_japanese,synopsis,title_english
0,Hanasaku Iroha,9289,https://myanimelist.net/anime/9289/Hanasaku_Iroha,https://cdn.myanimelist.net/images/anime/3/289...,2,26.0,PG-13 - Teens 13 or older,2020-09-13 11:23:05.509374,花咲くいろは,Ohana Matsumae is an energetic and wild teenag...,Hanasaku Iroha: Blossoms for Tomorrow
1,Uchuu Kyoudai,12431,https://myanimelist.net/anime/12431/Uchuu_Kyoudai,https://cdn.myanimelist.net/images/anime/7/375...,2,99.0,PG-13 - Teens 13 or older,2020-09-13 12:58:18.633841,宇宙兄弟,"On a fateful summer night in 2006, Mutta Nanba...",Space Brothers
2,Yozakura Quartet: Hana no Uta,18497,https://myanimelist.net/anime/18497/Yozakura_Q...,https://cdn.myanimelist.net/images/anime/5/525...,2,13.0,PG-13 - Teens 13 or older,2020-09-13 14:39:45.15066,夜桜四重奏 ～ハナノウタ～,"Hundreds of years ago, the borders between the...",NaN
3,Glass no Kamen Desu ga to Z Specials,34336,https://myanimelist.net/anime/34336/Glass_no_K...,https://cdn.myanimelist.net/images/anime/13/82...,2,2.0,PG-13 - Teens 13 or older,2020-09-14 01:53:48.51646,ガラスの仮面ですが と Z,Two unaired episodes of Glass no Kamen Desu ga...,NaN
4,Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season,39587,https://myanimelist.net/anime/39587/Re_Zero_ka...,https://cdn.myanimelist.net/images/anime/1444/...,1,13.0,R - 17+ (violence & profanity),2020-09-14 06:45:31.052373,Re:ゼロから始める異世界生活 2nd season,"Even after dying countless times, Subaru final...",Re:ZERO -Starting Life in Another World- Season 2


In [ ]:
anime_df.columns

In [ ]:
anime_df.shape

In [ ]:
anime_df.info()

In [ ]:
anime_df.isnull().sum()

In [ ]:
anime_df.columns = anime_df.columns.str.lower().str.strip()
anime_df.columns

In [ ]:
anime_df = anime_df[['title', 'title_english', 'synopsis', 'airing_status', 'mpaa_rating']]
anime_df.head()


In [ ]:
anime_df = anime_df.dropna(subset=['synopsis'])

anime_df['title_english'] = anime_df['title_english'].fillna('')
anime_df['airing_status'] = anime_df['airing_status'].fillna('')
anime_df['mpaa_rating'] = anime_df['mpaa_rating'].fillna('')


In [ ]:
anime_df['synopsis'] = anime_df['synopsis'].apply(lambda x: x.lower())
anime_df['title'] = anime_df['title'].apply(lambda x: x.lower())
anime_df['title_english'] = anime_df['title_english'].apply(lambda x: x.lower())


In [ ]:
anime_df['synopsis'] = anime_df['synopsis'].astype(str)
anime_df['title'] = anime_df['title'].astype(str)
anime_df['title_english'] = anime_df['title_english'].astype(str)
anime_df['airing_status'] = anime_df['airing_status'].astype(str)
anime_df['mpaa_rating'] = anime_df['mpaa_rating'].astype(str)


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

anime_df['synopsis'] = anime_df['synopsis'].apply(clean_text)

In [ ]:
anime_df['tags'] = (
    anime_df['title'] + " " +
    anime_df['title_english'] + " " +
    anime_df['synopsis']
)


In [ ]:
anime_df = anime_df[['title', 'title_english', 'synopsis', 'tags']]
anime_df.head()


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
STOP_WORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'has', 'he', 'her', 'his', 'in',
    'is', 'it', 'its', 'of', 'on', 'or', 'she', 'that', 'the', 'their', 'them', 'they', 'this',
    'to', 'was', 'were', 'when', 'where', 'who', 'with', 'after', 'before', 'into', 'over',
    'about', 'while', 'which', 'being', 'been', 'will', 'can', 'one', 'two', 'new', 'now',
    'young', 'life', 'world', 'time', 'day', 'find', 'must', 'however', 'become', 'anime',
    'series', 'movie', 'season', 'special', 'episode', 'ova', 'ona', 'some', 'also', 'because',
    'could', 'would', 'should', 'might', 'many', 'every', 'each', 'even', 'only', 'both', 'such',
    'through', 'together', 'during', 'more', 'most', 'once', 'upon', 'known', 'what', 'against',
    'these', 'those', 'alongside', 'another', 'around'
}

FRANCHISE_WORDS = {
    'season', 'movie', 'special', 'ova', 'ona', 'part', 'series', 'second', 'third', 'final',
    'shippuuden', 'shippuden', 'kai', 'the', 'a', 'an'
}

LOW_SIGNAL_WORDS = {
    'school', 'hospital', 'doctor', 'doctors', 'student', 'students', 'girl', 'boy', 'city',
    'town', 'village', 'friend', 'friends', 'family', 'father', 'mother', 'room', 'club',
    'team', 'work', 'working', 'daily', 'living', 'lives', 'begin', 'begins', 'comes',
    'instead', 'position', 'director', 'inside', 'across', 'high'
}

LOW_SIGNAL_PHRASES = {
    'high school', 'school life', 'young girl', 'young boy', 'hospital director', 'daily life'
}

THEME_RULES = {
    'psychological': {
        'psychological', 'mind', 'conscience', 'trauma', 'identity', 'memory', 'mystery',
        'detective', 'murder', 'killer', 'criminal', 'crime', 'pursuit', 'monster', 'genius',
        'strategy', 'manipulation', 'suspense'
    },
    'crime': {
        'crime', 'criminal', 'detective', 'murder', 'killer', 'police', 'investigation',
        'case', 'justice', 'pursuit', 'evidence', 'victim', 'law', 'trial', 'conspiracy'
    },
    'war': {
        'war', 'military', 'army', 'soldier', 'battle', 'battlefield', 'oppression', 'survival',
        'walls', 'invasion', 'rebellion', 'resistance', 'humanity', 'weapon', 'corps'
    },
    'romance': {
        'romance', 'romantic', 'love', 'fate', 'heart', 'relationship', 'couple', 'feelings',
        'emotional', 'dream', 'body', 'swap', 'supernatural', 'confession', 'date'
    },
    'sci-fi': {
        'scientific', 'science', 'future', 'technology', 'experiment', 'lab', 'time', 'travel',
        'space', 'robot', 'mecha', 'dimension', 'microwave', 'hacker', 'machine', 'alien'
    },
    'horror': {
        'horror', 'ghoul', 'gore', 'blood', 'flesh', 'monster', 'demon', 'vampire', 'curse',
        'fear', 'creature', 'supernatural', 'dark', 'terror', 'ghost'
    },
    'action': {
        'action', 'battle', 'fight', 'fighting', 'combat', 'power', 'warrior', 'soldier',
        'army', 'military', 'weapon', 'attack', 'mission', 'survival'
    },
    'adventure': {
        'adventure', 'journey', 'travel', 'crew', 'pirate', 'treasure', 'island', 'quest',
        'explore', 'world', 'ship', 'sea', 'grand', 'kingdom'
    },
    'fantasy': {
        'fantasy', 'magic', 'magical', 'demon', 'dragon', 'kingdom', 'curse', 'spirit',
        'supernatural', 'wizard', 'sword', 'myth', 'god', 'beast'
    },
    'historical': {
        'historical', 'history', 'era', 'samurai', 'warrior', 'viking', 'medieval', 'ancient',
        'kingdom', 'empire', 'japan', 'england', 'denmark', 'thorfinn', 'revenge'
    },
    'thriller': {
        'thriller', 'suspense', 'mystery', 'death', 'chase', 'danger', 'conspiracy', 'secret',
        'murder', 'killer', 'survival', 'terror', 'mind', 'game'
    }
}

THEME_WEIGHTS = {theme: 3.0 for theme in THEME_RULES}

THEME_PRIORITY = {
    'psychological': 3,
    'crime': 3,
    'war': 3,
    'adventure': 3,
    'historical': 3,
    'thriller': 2,
    'action': 2,
    'sci-fi': 2,
    'fantasy': 1,
    'horror': 1,
    'romance': 2
}

CONCEPT_MAP = {
    'attack on titan': ['war', 'survival', 'oppression', 'humanity', 'military', 'walls'],
    'shingeki no kyojin': ['war', 'survival', 'oppression', 'humanity', 'military', 'walls'],
    'death note': ['mind game', 'moral conflict', 'intelligence', 'psychological', 'crime', 'justice'],
    'one piece': ['freedom', 'adventure', 'crew', 'pirate', 'treasure', 'grand line']
}

HIGH_SIGNAL_WORDS = set().union(*THEME_RULES.values()).union({
    'ninja', 'ghoul', 'titan', 'shinobi', 'jutsu', 'hokage', 'shinigami'
})

MEDIUM_SIGNAL_WORDS = {
    'death', 'game', 'past', 'curse', 'spirit', 'secret', 'surgery', 'mysterious',
    'phenomenon', 'body', 'fate', 'training', 'rival', 'academy'
}

HIGH_SIGNAL_PHRASES = {
    'time travel', 'mind game', 'dark fantasy', 'serial killer', 'urban fantasy',
    'parallel world', 'body swap', 'space pirate', 'death note', 'tokyo ghoul', 'steins gate',
    'strange phenomenon', 'dream life', 'taki body', 'mysterious death', 'criminal past',
    'crime drama', 'psychological thriller', 'world war', 'civil war', 'dark fantasy',
    'grand line', 'straw hat', 'survey corps', 'humanity survival', 'viking war'
}

MEDIUM_SIGNAL_PHRASES = {
    'secret organization', 'special ability', 'demon king', 'main character', 'twist fate',
    'future gadgets', 'phone microwave', 'supernatural horror', 'military police', 'treasure hunt'
}

CORE_FRANCHISE_TERMS = {
    'naruto', 'gundam', 'fate', 'bleach', 'boruto', 'pokemon', 'digimon', 'dragonball',
    'dragon', 'conan', 'precure', 'sailor', 'jojo', 'monogatari', 'evangelion', 'ghoul',
    'steins', 'kyojin', 'titan', 'piece', 'vinland'
}

SIMILARITY_THRESHOLD = 0.4
KEYWORD_SCORE_THRESHOLD = 0.05
THEME_CONFIDENCE_THRESHOLD = 2
MAX_QUERY_THEMES = 3
CORE_THEME_COUNT = 2
KEYWORD_DOCUMENT_FREQUENCY = {}

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z ]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def tokenize_text(text):
    return normalize_text(text).split()

def get_candidate_words(text):
    return [
        word for word in tokenize_text(text)
        if len(word) >= 4 and word not in STOP_WORDS
    ]

def get_candidate_bigrams(text):
    words = [word for word in tokenize_text(text) if len(word) > 2 and word not in STOP_WORDS]
    return [
        f'{first_word} {second_word}'
        for first_word, second_word in zip(words, words[1:])
        if first_word != second_word
    ]

def build_document_frequency():
    document_frequency = {}

    for tags in anime_df['tags']:
        document_terms = set(get_candidate_words(tags)).union(get_candidate_bigrams(tags))
        for term in document_terms:
            document_frequency[term] = document_frequency.get(term, 0) + 1

    return document_frequency

def get_rarity_score(term):
    frequency = KEYWORD_DOCUMENT_FREQUENCY.get(normalize_text(term), 1)
    return 1 / frequency

def get_base_keyword_weight(term, themes=None):
    term = normalize_text(term)
    themes = themes or []

    if term in LOW_SIGNAL_PHRASES:
        return 0.3
    if term in HIGH_SIGNAL_PHRASES:
        return 3.5
    if term in MEDIUM_SIGNAL_PHRASES:
        return 2.0
    if ' ' in term:
        word_weights = [get_base_keyword_weight(word, themes) for word in term.split()]
        return min(3.5, max(0.4, sum(word_weights) / len(word_weights) + 0.3))
    if term in LOW_SIGNAL_WORDS:
        return 0.25
    if any(term in THEME_RULES.get(theme, set()) for theme in themes):
        return 3.0
    if term in HIGH_SIGNAL_WORDS:
        return 2.5
    if term in MEDIUM_SIGNAL_WORDS:
        return 1.5
    return 1.0

def get_keyword_weight(term, themes=None, use_rarity=True):
    base_weight = get_base_keyword_weight(term, themes)
    if not use_rarity:
        return base_weight
    rarity_score = get_rarity_score(term)
    return base_weight * (1 + rarity_score)

def extract_keyword_counts(text, excluded_words=None):
    excluded_words = excluded_words or set()
    keyword_counts = {}

    for word in get_candidate_words(text):
        if word in excluded_words:
            continue
        keyword_counts[word] = keyword_counts.get(word, 0) + 1

    return keyword_counts

def remove_dominant_noise_keywords(keyword_counts, max_removed=2):
    generic_candidates = [
        (word, count) for word, count in keyword_counts.items()
        if word in LOW_SIGNAL_WORDS
    ]
    generic_candidates = sorted(generic_candidates, key=lambda item: (-item[1], item[0]))
    removed_keywords = [word for word, _ in generic_candidates[:max_removed]]

    for word in removed_keywords:
        keyword_counts.pop(word, None)

    return removed_keywords

def extract_bigrams(text, max_bigrams=10, themes=None):
    bigram_counts = {}

    for bigram in get_candidate_bigrams(text):
        bigram_counts[bigram] = bigram_counts.get(bigram, 0) + 1

    ranked_bigrams = sorted(
        bigram_counts,
        key=lambda bigram: (-(bigram_counts[bigram] * get_keyword_weight(bigram, themes)), -bigram_counts[bigram], bigram)
    )
    return ranked_bigrams[:max_bigrams]

def extract_keywords(text, max_keywords=15, excluded_words=None, themes=None):
    keyword_counts = extract_keyword_counts(text, excluded_words)
    ranked_keywords = sorted(
        keyword_counts,
        key=lambda word: (-(keyword_counts[word] * get_keyword_weight(word, themes)), -keyword_counts[word], word)
    )
    return ranked_keywords[:max_keywords]

def build_keyword_pattern(terms):
    return '|'.join(re.escape(term) for term in terms)

def weighted_term_score(terms, candidate_text, themes=None):
    normalized_candidate = normalize_text(candidate_text)
    matched_terms = []
    matched_weight = 0
    total_possible_weight = sum(get_keyword_weight(term, themes) for term in terms)

    if total_possible_weight == 0:
        return 0, matched_terms

    for term in terms:
        normalized_term = normalize_text(term)
        if re.search(r'\b' + re.escape(normalized_term) + r'\b', normalized_candidate):
            matched_terms.append(term)
            matched_weight += get_keyword_weight(term, themes)

    return matched_weight / total_possible_weight, matched_terms

def score_themes_from_terms(terms):
    theme_scores = []

    for theme, theme_words in THEME_RULES.items():
        score = 0
        matched_theme_terms = []

        for term in terms:
            normalized_term = normalize_text(term)
            words = set(normalized_term.split())
            overlap = words.intersection(theme_words)
            if overlap:
                term_score = sum(get_base_keyword_weight(word, [theme]) for word in overlap)
                score += term_score
                matched_theme_terms.extend(sorted(overlap))
            if normalized_term in HIGH_SIGNAL_PHRASES:
                score += 1.5
                matched_theme_terms.append(normalized_term)

        if score > 0:
            priority = THEME_PRIORITY.get(theme, 1)
            theme_scores.append({
                'theme': theme,
                'raw_score': score,
                'priority': priority,
                'score': score * priority,
                'matched_terms': list(dict.fromkeys(matched_theme_terms))
            })

    return sorted(theme_scores, key=lambda item: (-item['score'], item['theme']))

def build_theme_profile(terms, max_themes=MAX_QUERY_THEMES, core_count=CORE_THEME_COUNT):
    ranked_theme_scores = score_themes_from_terms(terms)

    if not ranked_theme_scores:
        return {
            'ranked': [],
            'themes': [],
            'core': [],
            'secondary': [],
            'weak': [],
            'scores': {}
        }

    strongest_score = ranked_theme_scores[0]['score']
    confidence_floor = max(THEME_CONFIDENCE_THRESHOLD, strongest_score * 0.4)
    confident_themes = [item for item in ranked_theme_scores if item['score'] >= confidence_floor]
    selected_themes = confident_themes[:max_themes]
    weak_themes = [
        item['theme'] for item in ranked_theme_scores
        if item['theme'] not in {selected['theme'] for selected in selected_themes}
    ]
    themes = [item['theme'] for item in selected_themes]
    core_themes = themes[:core_count]
    secondary_themes = themes[core_count:]

    return {
        'ranked': ranked_theme_scores,
        'themes': themes,
        'core': core_themes,
        'secondary': secondary_themes,
        'weak': weak_themes,
        'scores': {item['theme']: item['score'] for item in ranked_theme_scores}
    }

def detect_query_theme_profile(keyword_counts, bigrams):
    terms = set(keyword_counts).union(normalize_text(bigram) for bigram in bigrams)
    return build_theme_profile(terms)

def get_query_theme_terms(query_themes, keyword_counts, bigrams):
    available_terms = set(keyword_counts).union(normalize_text(bigram) for bigram in bigrams)
    theme_terms = []

    for theme in query_themes:
        for term in sorted(THEME_RULES[theme]):
            if term in available_terms:
                theme_terms.append(term)

    return list(dict.fromkeys(theme_terms))

def detect_candidate_theme_profile(candidate_text):
    terms = set(get_candidate_words(candidate_text)).union(get_candidate_bigrams(candidate_text))
    return build_theme_profile(terms)

def weighted_theme_score(core_themes, secondary_themes, weak_themes, candidate_themes):
    candidate_theme_set = set(candidate_themes)

    core_match = len(set(core_themes).intersection(candidate_theme_set)) / len(core_themes) if core_themes else 0
    secondary_match = len(set(secondary_themes).intersection(candidate_theme_set)) / len(secondary_themes) if secondary_themes else 0
    weak_match = len(set(weak_themes).intersection(candidate_theme_set)) / len(weak_themes) if weak_themes else 0

    return (0.6 * core_match) + (0.3 * secondary_match) + (0.1 * weak_match), core_match, secondary_match, weak_match

def get_theme_match_type(core_themes, secondary_themes, candidate_themes):
    candidate_theme_set = set(candidate_themes)
    if set(core_themes).intersection(candidate_theme_set):
        return 'core'
    if set(secondary_themes).intersection(candidate_theme_set):
        return 'secondary'
    return 'none'

def theme_filter_candidates(core_themes, secondary_themes):
    if not core_themes:
        return anime_df.copy(), 'None'

    theme_terms = sorted(set().union(*(THEME_RULES[theme] for theme in core_themes)))
    theme_pattern = build_keyword_pattern(theme_terms)
    filtered_df = anime_df[anime_df['tags'].str.contains(theme_pattern, case=False, na=False, regex=True)].copy()

    if len(filtered_df) < 100:
        return anime_df.copy(), 'soft core-theme fallback: too few theme matches'

    return filtered_df, ', '.join(core_themes)

def filter_candidates(query_keywords, query_bigrams, core_themes, secondary_themes, min_size=500, max_size=5000):
    theme_df, theme_filter = theme_filter_candidates(core_themes, secondary_themes)
    normalized_tags = theme_df['tags'].apply(normalize_text)
    base_terms = list(dict.fromkeys(query_bigrams[:8] + query_keywords[:12]))
    broad_terms = list(dict.fromkeys(query_bigrams + query_keywords))

    if not base_terms:
        return theme_df.copy(), '', [], f'core-theme filter only: {theme_filter}', 0, theme_filter

    pattern = build_keyword_pattern(base_terms)
    filtered_df = theme_df[normalized_tags.str.contains(pattern, case=False, na=False, regex=True)].copy()
    filter_mode = 'core-theme + broad weighted term match'
    filter_weight_threshold = 0

    if len(filtered_df) > max_size:
        scored_rows = []
        filter_weight_threshold = 2.5
        active_themes = core_themes + secondary_themes

        for row_idx in filtered_df.index:
            matched_weight = 0
            normalized_row_tags = normalize_text(anime_df.loc[row_idx, 'tags'])
            for term in base_terms:
                if re.search(r'\b' + re.escape(normalize_text(term)) + r'\b', normalized_row_tags):
                    matched_weight += get_base_keyword_weight(term, active_themes)
            if matched_weight >= filter_weight_threshold:
                scored_rows.append(row_idx)

        tightened_df = anime_df.loc[scored_rows].copy()
        if len(tightened_df) >= min_size:
            filtered_df = tightened_df
            filter_mode = 'core-theme + tightened weighted term match'

    if len(filtered_df) < min_size:
        broad_pattern = build_keyword_pattern(broad_terms)
        broadened_df = theme_df[normalized_tags.str.contains(broad_pattern, case=False, na=False, regex=True)].copy()

        if len(broadened_df) >= 50:
            filtered_df = broadened_df
            pattern = broad_pattern
            filter_mode = 'core-theme + loosened broad term match'
            filter_weight_threshold = 0
        else:
            filtered_df = theme_df.copy()
            pattern = ''
            filter_mode = f'core-theme fallback only: {theme_filter}'
            filter_weight_threshold = 0

    return filtered_df, pattern, base_terms, filter_mode, filter_weight_threshold, theme_filter

def build_query_franchise_keys(anime_row):
    title_text = normalize_text(f"{anime_row['title']} {anime_row['title_english']}")
    words = [
        word for word in title_text.split()
        if len(word) > 3 and word not in STOP_WORDS and word not in FRANCHISE_WORDS
    ]
    keys = set()

    for title_value in [anime_row['title'], anime_row['title_english']]:
        normalized_title = normalize_text(title_value)
        if normalized_title:
            keys.add(normalized_title)

    for word in words:
        if word in CORE_FRANCHISE_TERMS:
            keys.add(word)

    if len(words) >= 2:
        keys.add(' '.join(words[:2]))

    return keys

def get_franchise_key(title, query_franchise_keys=None):
    normalized_title = normalize_text(title)
    query_franchise_keys = query_franchise_keys or set()

    for key in sorted(query_franchise_keys, key=len, reverse=True):
        if key and re.search(r'\b' + re.escape(key) + r'\b', normalized_title):
            return key

    title_head = re.split(r':|-|\bseason\b|\bmovie\b|\bspecial\b|\bova\b|\bona\b|\bpart\b', str(title).lower())[0]
    head_words = [word for word in normalize_text(title_head).split() if word not in FRANCHISE_WORDS]

    for word in head_words:
        if word in CORE_FRANCHISE_TERMS:
            return word

    return ' '.join(head_words[:2]) if head_words else normalized_title

def find_matching_anime(query):
    normalized_query = normalize_text(query)
    normalized_titles = anime_df['title'].apply(normalize_text)
    normalized_english_titles = anime_df['title_english'].apply(normalize_text)

    matches = anime_df[
        normalized_titles.str.contains(normalized_query, case=False, na=False, regex=False) |
        normalized_english_titles.str.contains(normalized_query, case=False, na=False, regex=False)
    ]

    if matches.empty:
        return None

    exact_matches = matches[
        normalized_titles.loc[matches.index].eq(normalized_query) |
        normalized_english_titles.loc[matches.index].eq(normalized_query)
    ]
    selected_matches = exact_matches if not exact_matches.empty else matches
    return selected_matches.index[0]

def format_rarity_adjusted_terms(terms, themes):
    return ', '.join(
        f'{term} ({get_keyword_weight(term, themes):.4f})'
        for term in terms
    )

def get_concept_keywords(anime_row, user_query):
    lookup_values = [
        normalize_text(user_query),
        normalize_text(anime_row['title']),
        normalize_text(anime_row['title_english'])
    ]

    for value in lookup_values:
        if value in CONCEPT_MAP:
            return CONCEPT_MAP[value]

    return []

def is_same_franchise(candidate_title, query_franchise_keys):
    normalized_candidate = normalize_text(candidate_title)
    return any(
        key and re.search(r'\b' + re.escape(key) + r'\b', normalized_candidate)
        for key in query_franchise_keys
    )

def format_theme_scores(theme_profile):
    if not theme_profile['ranked']:
        return 'None'
    return ', '.join(
        f"{item['theme']}={item['score']:.2f} (raw={item['raw_score']:.2f}, priority={item['priority']})"
        for item in theme_profile['ranked']
    )

def recommend_anime(
    title,
    top_n=5,
    max_same_series=2,
    similarity_threshold=SIMILARITY_THRESHOLD,
    keyword_score_threshold=KEYWORD_SCORE_THRESHOLD
):
    idx = find_matching_anime(title)

    if idx is None:
        print('Anime not found')
        return []

    anime_row = anime_df.loc[idx]
    query_tags = anime_row['tags']
    raw_keyword_counts = extract_keyword_counts(query_tags)
    removed_keywords = remove_dominant_noise_keywords(raw_keyword_counts)
    preliminary_bigrams = extract_bigrams(query_tags, max_bigrams=12)
    concept_keywords = get_concept_keywords(anime_row, title)
    for concept_keyword in concept_keywords:
        for concept_word in normalize_text(concept_keyword).split():
            if len(concept_word) >= 4 and concept_word not in STOP_WORDS:
                raw_keyword_counts[concept_word] = raw_keyword_counts.get(concept_word, 0) + 2
    preliminary_bigrams = list(dict.fromkeys(concept_keywords + preliminary_bigrams))
    theme_profile = detect_query_theme_profile(raw_keyword_counts, preliminary_bigrams)
    query_themes = theme_profile['themes']
    core_themes = theme_profile['core']
    secondary_themes = theme_profile['secondary']
    weak_themes = theme_profile['weak']
    active_themes = core_themes + secondary_themes
    query_theme_terms = get_query_theme_terms(query_themes, raw_keyword_counts, preliminary_bigrams)
    query_keywords = extract_keywords(query_tags, max_keywords=20, themes=active_themes)
    query_keywords = list(dict.fromkeys(concept_keywords + [keyword for keyword in query_keywords if keyword not in removed_keywords]))
    query_bigrams = extract_bigrams(query_tags, max_bigrams=12, themes=active_themes)
    query_bigrams = [bigram for bigram in query_bigrams if bigram not in LOW_SIGNAL_PHRASES]
    scoring_terms = list(dict.fromkeys(concept_keywords + query_theme_terms + query_bigrams[:6] + query_keywords[:12]))
    query_franchise_keys = build_query_franchise_keys(anime_row)

    filtered_df, pattern, filter_terms, filter_mode, filter_weight_threshold, theme_filter = filter_candidates(
        query_keywords,
        query_bigrams,
        core_themes,
        secondary_themes
    )
    filtered_df = filtered_df[filtered_df.index != idx]

    print('Matched anime:', anime_row['title'])
    print('Detected Themes (ranked):', ', '.join(query_themes) if query_themes else 'general')
    print('Core Themes:', ', '.join(core_themes) if core_themes else 'None')
    print('Secondary Themes:', ', '.join(secondary_themes) if secondary_themes else 'None')
    print('Removed Weak Themes:', ', '.join(weak_themes) if weak_themes else 'None')
    print('Theme Confidence Scores:', format_theme_scores(theme_profile))
    print('Adjusted Theme Scores:', format_theme_scores(theme_profile))
    print('Theme Priority Applied:', THEME_PRIORITY)
    print('Concept Keywords Added:', ', '.join(concept_keywords) if concept_keywords else 'None')
    print('Removed low-signal words:', ', '.join(removed_keywords) if removed_keywords else 'None')
    print('Extracted keywords:', ', '.join(query_keywords))
    print('Extracted bigrams:', ', '.join(query_bigrams))
    print('Theme terms:', ', '.join(query_theme_terms) if query_theme_terms else 'None')
    print('Rarity-adjusted keywords:', format_rarity_adjusted_terms(scoring_terms, active_themes))
    print('Final keyword list:', ', '.join(scoring_terms))
    print('Theme filter:', theme_filter)
    print('Filter mode:', filter_mode)
    print('Regex pattern:', pattern if pattern else 'None')
    print('Filtered dataset size:', len(filtered_df), 'of', len(anime_df))
    print('Filter weight threshold:', filter_weight_threshold)
    print('Similarity threshold:', similarity_threshold)
    print('Keyword score threshold:', keyword_score_threshold)

    if filtered_df.empty:
        print('No recommendations found')
        return []

    query_vector = model.encode([anime_row['tags']])
    filtered_vectors = model.encode(filtered_df['tags'].tolist(), show_progress_bar=True)
    similarity_scores = cosine_similarity(query_vector, filtered_vectors)[0]

    scored_recommendations = []
    for position, (_, candidate) in enumerate(filtered_df.iterrows()):
        similarity_score = float(similarity_scores[position])
        if similarity_score < similarity_threshold:
            continue

        keyword_score, matched_terms = weighted_term_score(scoring_terms, candidate['tags'], active_themes)
        if keyword_score < keyword_score_threshold:
            continue

        candidate_profile = detect_candidate_theme_profile(candidate['tags'])
        candidate_themes = candidate_profile['themes']

        theme_score, core_match, secondary_match, weak_match = weighted_theme_score(
            core_themes,
            secondary_themes,
            weak_themes,
            candidate_themes
        )
        theme_match_type = get_theme_match_type(core_themes, secondary_themes, candidate_themes)
        boosted_keyword_score = keyword_score ** 1.5
        final_score = (0.5 * similarity_score) + (0.35 * boosted_keyword_score) + (0.15 * theme_score)
        soft_theme_penalty_applied = theme_match_type != 'core'
        if soft_theme_penalty_applied:
            final_score *= 0.75

        same_franchise = is_same_franchise(
            f"{candidate['title']} {candidate['title_english']}",
            query_franchise_keys
        )
        if same_franchise:
            final_score *= 0.8

        scored_recommendations.append({
            'title': candidate['title'],
            'similarity_score': similarity_score,
            'keyword_score': keyword_score,
            'boosted_keyword_score': boosted_keyword_score,
            'theme_score': theme_score,
            'core_match': core_match,
            'secondary_match': secondary_match,
            'weak_match': weak_match,
            'theme_match_type': theme_match_type,
            'soft_theme_penalty_applied': soft_theme_penalty_applied,
            'same_franchise': same_franchise,
            'final_score': final_score,
            'matched_terms': matched_terms,
            'candidate_themes': candidate_themes,
            'candidate_theme_scores': candidate_profile,
            'tags': candidate['tags'],
            'franchise_key': get_franchise_key(
                f"{candidate['title']} {candidate['title_english']}",
                query_franchise_keys
            )
        })

    scored_recommendations = sorted(scored_recommendations, key=lambda item: item['final_score'], reverse=True)
    recommendations = []
    franchise_counts = {}

    for recommendation in scored_recommendations:
        franchise_key = recommendation['franchise_key']
        if franchise_counts.get(franchise_key, 0) >= max_same_series:
            continue

        recommendations.append(recommendation)
        franchise_counts[franchise_key] = franchise_counts.get(franchise_key, 0) + 1

        print('Title:', recommendation['title'])
        print('Candidate Themes:', ', '.join(recommendation['candidate_themes']) if recommendation['candidate_themes'] else 'general')
        print('Theme Match Type:', recommendation['theme_match_type'])
        print('Theme Score:', round(recommendation['theme_score'], 4))
        print('Theme Components:', 'core=', round(recommendation['core_match'], 4), 'secondary=', round(recommendation['secondary_match'], 4), 'weak=', round(recommendation['weak_match'], 4))
        print('Keyword Score:', round(recommendation['keyword_score'], 4))
        print('Boosted Keyword Score:', round(recommendation['boosted_keyword_score'], 4))
        print('Similarity Score:', round(recommendation['similarity_score'], 4))
        print('Soft Theme Penalty Applied:', 'yes' if recommendation['soft_theme_penalty_applied'] else 'no')
        print('Franchise Penalty Applied:', 'yes' if recommendation['same_franchise'] else 'no')
        print('Final Score Breakdown:', '0.5*similarity + 0.35*(keyword^1.5) + 0.15*theme, then penalties')
        print('Final Score:', round(recommendation['final_score'], 4))
        print('Matched Keywords:', ', '.join(recommendation['matched_terms']))
        print('Tags:', recommendation['tags'][:100])
        print()

        if len(recommendations) == top_n:
            break

    if not recommendations:
        print('No recommendations met the similarity, keyword-score, and core-theme thresholds')

    return recommendations

KEYWORD_DOCUMENT_FREQUENCY = build_document_frequency()


In [ ]:
test_titles = ['attack on titan', 'death note']

for test_title in test_titles:
    print('=' * 80)
    print('Recommendations for:', test_title)
    recommend_anime(test_title)
